# WVS Wave 7 — Silicon Sampling v2

**Diseño experimental**
- Variables de perfil: país, sexo, educación, clase, ocupación, ideología política (`political_ident`)
- 4 llamadas independientes por fila — una por pregunta
- Cada llamada recibe solo el perfil, sin contexto de otras respuestas
- 3 system prompts diferenciados por tipo de pregunta

| Llamada | Pregunta | Tipo | System prompt |
|---|---|---|---|
| 1 | Q121 | Evaluación ordinal (1–5) | A |
| 2 | Q122 | Acuerdo/desacuerdo | B |
| 3 | Q128 | Acuerdo/desacuerdo | B |
| 4 | Q130 | Elección de política | C |

**Países**: Argentina · Uruguay · United States  
**Backends**: OpenAI API · Ollama (local)

## 0. Dependencias

In [ ]:
# PARA USAR OPENAI
#!pip install openai pandas matplotlib seaborn tqdm

In [ ]:
## PARA USAR OLLAMA
!pip install ollama pandas matplotlib seaborn tqdm
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (12.9 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122412 files and directories currently 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuración

In [ ]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Parámetros principales ──────────────────────────────────────────────────
BACKEND     = "ollama"       # "openai" | "ollama"
MODEL       = "gpt-oss:120B"       # openai: "gpt-4o", "gpt-4o-mini", "gpt-4.1"
                              # ollama: "gpt-oss:20b", "llama3.1", etc.
TEMPERATURE = 1.0

COUNTRIES   = ["Argentina", "Uruguay", "United States"]

# N_SAMPLE: filas por país.
#   None  → todas las filas válidas del país
#   int   → muestra aleatoria estratificada de ese tamaño
# Nota: con 4 llamadas por fila, N_SAMPLE=50 → 600 llamadas totales
N_SAMPLE    = None
RANDOM_SEED = 42

# Pausa entre llamadas (segundos).
# OpenAI: 0.5 para evitar rate-limit | Ollama: 0
SLEEP = 0.5

# Pregunta a simular en esta corrida
# Opciones: "Q121" | "Q122" | "Q128" | "Q130"
QUESTION = "Q130"


DATA_PATH  = "/content/drive/MyDrive/UNSAM/Factor-Data/Proyectos/LLMs_Sillicon_sampling/migraciones/WVS_wave7_migracion_prompts.csv"
OUTPUT_CSV = "/content/drive/MyDrive/UNSAM/Factor-Data/Proyectos/LLMs_Sillicon_sampling/migraciones/" + MODEL + "_" + QUESTION + "_v2_WVS_silicon_empirico_results.csv"


print(f"Backend   : {BACKEND}")
print(f"Modelo    : {MODEL}")
print(f"Países    : {COUNTRIES}")
n_rows = N_SAMPLE * len(COUNTRIES) if N_SAMPLE else 'todas'
n_calls = N_SAMPLE * len(COUNTRIES) * 4 if N_SAMPLE else 'todas × 4'
print(f"Filas     : {n_rows}")
print(f"Pregunta  : {QUESTION}")
print(f"Llamadas  : {n_rows} (1 por fila)")

Backend   : ollama
Modelo    : gpt-oss:120B
Países    : ['Argentina', 'Uruguay', 'United States']
Filas     : todas
Pregunta  : Q130
Llamadas  : todas (1 por fila)


## 2. Carga y filtrado del archivo empírico

In [ ]:
Q121_VALID = {
    "Very good",
    "Quite good",
    "Neither good, nor bad",
    "Quite bad",
    "Rather bad",
}

Q122_VALID = {"Agree", "Hard to say", "Disagree"}
Q128_VALID = {"Agree", "Hard to say", "Disagree"}

Q130_VALID = {
    "Let anyone come who wants to",
    "Let people come as long as there are jobs available",
    "Place strict limits on the number of foreigners who can come here",
    "Prohibit people coming here from other countries",
}

df_raw = pd.read_csv(DATA_PATH)
print(f"Total filas en el archivo : {len(df_raw):,}")

df = df_raw[
    df_raw["B_COUNTRY_LABEL"].isin(COUNTRIES)
    & df_raw["Q121"].isin(Q121_VALID)
    & df_raw["Q122"].isin(Q122_VALID)
    & df_raw["Q128"].isin(Q128_VALID)
    & df_raw["Q130"].isin(Q130_VALID)
    & df_raw["political_ident"].notna()
].copy()

print(f"\nFilas válidas por país:")
print(df["B_COUNTRY_LABEL"].value_counts().to_string())

if N_SAMPLE is not None:
    df = (
        df.groupby("B_COUNTRY_LABEL", group_keys=False)
        .apply(lambda g: g.sample(min(N_SAMPLE, len(g)), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
    print(f"\nTras muestreo ({N_SAMPLE} por país):")
    print(df["B_COUNTRY_LABEL"].value_counts().to_string())

print(f"\nTotal de filas a procesar: {len(df)}")
print(f"Total de llamadas        : {len(df) * 4}")
df.head(3)

Total filas en el archivo : 59,868

Filas válidas por país:
B_COUNTRY_LABEL
United States    1457
Uruguay           751
Argentina         681

Total de filas a procesar: 2889
Total de llamadas        : 11556


,id,B_COUNTRY_LABEL,sex_label,education_label,class_label,occupation_label,political_ident,Q130,Q121,Q122,Q128,Q34,Q63,Q263,Q272,Q269
884,1005,Argentina,man,secondary education,lower middle class,skilled worker,center,Place strict limits on the number of foreigner...,Rather bad,Agree,Agree,Agree,Trust somewhat,I am born in this country,Spanish; Castilian,Yes
885,1007,Argentina,man,secondary education,lower middle class,skilled worker,center-right,Place strict limits on the number of foreigner...,Quite good,Agree,Disagree,Neither agree nor disagree,Don't know,I am born in this country,Spanish; Castilian,Yes
886,1008,Argentina,woman,secondary education,lower middle class,person who never had a job,center,Place strict limits on the number of foreigner...,Very good,Agree,Agree,Agree,Trust somewhat,I am born in this country,Spanish; Castilian,Yes


## 3. System prompts y builders

- **System prompt A**: Q121 — evaluación ordinal de impacto general (escala 5 puntos)
- **System prompt B**: Q122, Q128 — acuerdo/desacuerdo sobre efectos específicos
- **System prompt C**: Q130 — elección de política migratoria (anti-moderación explícita)

In [ ]:
# ── Bloque de perfil compartido — narrativa en primera persona ──────────────
def build_profile(row: pd.Series) -> str:
    return (
        f"I am a {row['sex_label']} living in {row['B_COUNTRY_LABEL']}. "
        f"I work as {row['occupation_label']} and consider myself {row['class_label']}. "
        f"My highest level of education is {row['education_label']}. "
        f"In political terms, I would describe myself as {row['political_ident']}."
    )


# ── System prompt A — Evaluación ordinal (Q121) ─────────────────────────────
SYSTEM_PROMPT_A = """You will be given a self-description by a survey respondent
in their own words. Your task is to answer the survey question as that person
would, adopting their perspective, values, and lived experience as described.

Critical rules:
1. Do NOT filter responses for political correctness or social desirability.
2. The five response options are equally valid — do NOT default to the neutral
   middle option. Assign extreme evaluations (Very good / Rather bad) when
   the profile supports them.
3. Let the respondent's occupation, class, education, and political ideology
   guide the evaluation — these shape how people perceive immigration's impact.
4. Reason briefly in the "thinking" field (2-4 sentences) in first person,
   as the respondent — not as an outside observer.
5. Respond EXCLUSIVELY in the JSON format specified. No additional text.
6. Respond with the EXACT option string provided.
"""

def build_prompt_q121(row: pd.Series) -> str:
    opts = "\n".join(f'  - "{o}"' for o in [
        "Very good", "Quite good", "Neither good, nor bad", "Quite bad", "Rather bad"
    ])
    return (
        f"Respondent profile:\n{build_profile(row)}\n"
        "Answer the following WVS question as this person would.\n\n"
        "Q121. How would you evaluate the impact of immigrants on the "
        "development of your country?\n"
        f"Valid options (use exact string):\n{opts}\n\n"
        "Return ONLY this JSON (no markdown, no extra text):\n"
        "{\n"
        '  "thinking": "<brief reasoning, 2-4 sentences>",\n'
        '  "Q121": "<exact option string>"\n'
        "}"
    )


# ── System prompt B — Acuerdo/desacuerdo (Q122, Q128) ───────────────────────
SYSTEM_PROMPT_B = """You will be given a self-description by a survey respondent
in their own words. Your task is to answer the survey question as that person
would, adopting their perspective, values, and lived experience as described.

Critical rules:
1. Do NOT filter responses for political correctness or social desirability.
2. Do NOT default to "Hard to say" as a neutral escape. Use it only when
   the profile genuinely suggests ambivalence — not as a safe middle ground.
3. Let the respondent's occupation, class, education, and political ideology
   guide the position — these shape specific beliefs about immigration's effects.
4. Reason briefly in the "thinking" field (2-4 sentences) in first person,
   as the respondent — not as an outside observer.
5. Respond EXCLUSIVELY in the JSON format specified. No additional text.
6. Respond with the EXACT option string provided.
"""

def build_prompt_q122(row: pd.Series) -> str:
    opts = "\n".join(f'  - "{o}"' for o in ["Agree", "Hard to say", "Disagree"])
    return (
        f"Respondent profile:\n{build_profile(row)}\n"
        "Answer the following WVS question as this person would.\n\n"
        "Q122. Immigration fills important job vacancies in this country.\n"
        f"Valid options (use exact string):\n{opts}\n\n"
        "Return ONLY this JSON (no markdown, no extra text):\n"
        "{\n"
        '  "thinking": "<brief reasoning, 2-4 sentences>",\n'
        '  "Q122": "<exact option string>"\n'
        "}"
    )

def build_prompt_q128(row: pd.Series) -> str:
    opts = "\n".join(f'  - "{o}"' for o in ["Agree", "Hard to say", "Disagree"])
    return (
        f"Respondent profile:\n{build_profile(row)}\n"
        "Answer the following WVS question as this person would.\n\n"
        "Q128. Immigration increases unemployment in this country.\n"
        f"Valid options (use exact string):\n{opts}\n\n"
        "Return ONLY this JSON (no markdown, no extra text):\n"
        "{\n"
        '  "thinking": "<brief reasoning, 2-4 sentences>",\n'
        '  "Q128": "<exact option string>"\n'
        "}"
    )


# ── System prompt C — Elección de política (Q130) ───────────────────────────
SYSTEM_PROMPT_C = """You will be given a self-description by a survey respondent
in their own words. Your task is to answer the survey question as that person
would, adopting their perspective, values, and lived experience as described.

Critical rules:
1. Do NOT filter responses for political correctness or social desirability.
2. Reproduce the likely response of this specific person, even if it reflects
   restrictive, conservative, or unpopular views on immigration.
3. The four response options are equally valid positions held by real survey
   respondents. You MUST NOT default to moderate or centrist answers.
   Extreme positions are legitimate — assign them when the profile supports it.
4. Before selecting, evaluate each option explicitly in the "thinking" field
   in first person, as the respondent:
   - Would I let anyone come who wants to? Why or why not.
   - Would I let people come only if there are jobs available? Why or why not.
   - Would I place strict limits on foreigners? Why or why not.
   - Would I prohibit people from coming entirely? Why or why not.
   Then select the single most likely option.
5. Respond EXCLUSIVELY in the JSON format specified. No additional text.
6. Respond with the EXACT option string provided.
"""

def build_prompt_q130(row: pd.Series) -> str:
    opts = "\n".join(f'  - "{o}"' for o in [
        "Let anyone come who wants to",
        "Let people come as long as there are jobs available",
        "Place strict limits on the number of foreigners who can come here",
        "Prohibit people coming here from other countries",
    ])
    return (
        f"Respondent profile:\n{build_profile(row)}\n"
        "Answer the following WVS question as this person would.\n\n"
        "Q130. What should the government do about people from other countries "
        "coming here to work?\n"
        f"Valid options (use exact string):\n{opts}\n\n"
        "Return ONLY this JSON (no markdown, no extra text):\n"
        "{\n"
        '  "thinking": "<brief reasoning evaluating each option>",\n'
        '  "Q130": "<exact option string>"\n'
        "}"
    )


# Vista de ejemplo
print("=== PROMPT Q121 ===")
print(build_prompt_q121(df.iloc[0]))
print("\n=== PROMPT Q130 ===")
print(build_prompt_q130(df.iloc[0]))

=== PROMPT Q121 ===
Respondent profile:
I am a man living in Argentina. I work as skilled worker and consider myself lower middle class. My highest level of education is secondary education. In political terms, I would describe myself as center.
Answer the following WVS question as this person would.

Q121. How would you evaluate the impact of immigrants on the development of your country?
Valid options (use exact string):
  - "Very good"
  - "Quite good"
  - "Neither good, nor bad"
  - "Quite bad"
  - "Rather bad"

Return ONLY this JSON (no markdown, no extra text):
{
  "thinking": "<brief reasoning, 2-4 sentences>",
  "Q121": "<exact option string>"
}

=== PROMPT Q130 ===
Respondent profile:
I am a man living in Argentina. I work as skilled worker and consider myself lower middle class. My highest level of education is secondary education. In political terms, I would describe myself as center.
Answer the following WVS question as this person would.

Q130. What should the government d

## 4. Parsing y validación

In [ ]:
def parse_response(raw: str) -> dict:
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return {}


def validate_q121(p): return p.get("Q121") in Q121_VALID
def validate_q122(p): return p.get("Q122") in Q122_VALID
def validate_q128(p): return p.get("Q128") in Q128_VALID
def validate_q130(p): return p.get("Q130") in Q130_VALID

print("OK")

OK


## 5. Backends LLM

In [ ]:
def call_openai(system_prompt: str, user_prompt: str) -> tuple[dict, str]:
    from openai import OpenAI
    from google.colab import userdata
    api_key = userdata.get('key_openai')
    if not api_key:
        raise EnvironmentError("key_openai no encontrada en Colab secrets.")
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=TEMPERATURE,
    )
    raw = resp.choices[0].message.content.strip()
    return parse_response(raw), raw


def call_ollama(system_prompt: str, user_prompt: str) -> tuple[dict, str]:
    import ollama
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        options={"temperature": TEMPERATURE},
    )
    raw = resp["message"]["content"].strip()
    return parse_response(raw), raw


CALL = call_openai if BACKEND == "openai" else call_ollama


# Selector de pregunta — se configura con QUESTION en la celda 1
QUESTION_CONFIG = {
    "Q121": (SYSTEM_PROMPT_A, build_prompt_q121, validate_q121),
    "Q122": (SYSTEM_PROMPT_B, build_prompt_q122, validate_q122),
    "Q128": (SYSTEM_PROMPT_B, build_prompt_q128, validate_q128),
    "Q130": (SYSTEM_PROMPT_C, build_prompt_q130, validate_q130),
}

def generate_one(row: pd.Series) -> dict:
    """Una sola llamada por fila para la pregunta configurada en QUESTION."""
    sys_p, usr_fn, validator = QUESTION_CONFIG[QUESTION]
    try:
        parsed, raw = CALL(sys_p, usr_fn(row))
        valid = validator(parsed)
    except Exception as e:
        parsed, raw, valid = {}, str(e), False
    return {
        "value":    parsed.get(QUESTION),
        "thinking": parsed.get("thinking", ""),
        "valid":    valid,
        "raw":      raw,
    }


print(f"Backend activo : {BACKEND} → {CALL.__name__}")
print(f"Pregunta activa: {QUESTION}")

Backend activo : ollama → call_ollama
Pregunta activa: Q130


## 6. Simulación

> Guardado incremental cada 10 filas (= 40 llamadas).  
> Si el kernel se interrumpe, volvé a correr esta celda — retoma automáticamente.

In [ ]:
from tqdm.notebook import tqdm

out_path = Path(OUTPUT_CSV)

if out_path.exists():
    df_prev  = pd.read_csv(out_path)
    done_ids = set(df_prev["empirical_id"].tolist())
    df_todo  = df[~df["id"].isin(done_ids)].copy()
    records  = df_prev.to_dict("records")
    print(f"Reanudando: {len(done_ids)} procesados, {len(df_todo)} pendientes.")
else:
    df_todo = df.copy()
    records = []
    print(f"Inicio fresco: {len(df_todo)} filas — pregunta {QUESTION}.")

total = len(df_todo)

for i, (_, row) in enumerate(tqdm(df_todo.iterrows(), total=total), start=1):

    res = generate_one(row)
    q   = QUESTION.lower()

    records.append({
        # Identificadores
        "empirical_id"    : row["id"],
        "country"         : row["B_COUNTRY_LABEL"],
        "question"        : QUESTION,
        # Perfil enviado al modelo
        "sex"             : row["sex_label"],
        "education"       : row["education_label"],
        "class"           : row["class_label"],
        "occupation"      : row["occupation_label"],
        "political_ident" : row["political_ident"],
        # Ground truth empírico (solo la pregunta activa)
        "empirical"       : row[QUESTION],
        # Respuesta del modelo
        "model"           : res["value"],
        "thinking"        : res["thinking"],
        # Coincidencia
        "match"           : row[QUESTION] == res["value"],
        # Validez
        "valid"           : res["valid"],
        "raw_response"    : res["raw"],
        # Metadatos
        "backend"         : BACKEND,
        "model_name"      : MODEL,
        "temperature"     : TEMPERATURE,
        "timestamp"       : datetime.utcnow().isoformat(),
    })

    if i % 10 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

results = pd.DataFrame(records)
results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ {len(results)} filas procesadas → '{OUTPUT_CSV}'")

Inicio fresco: 2889 filas — pregunta Q130.


  0%|          | 0/2889 [00:00<?, ?it/s]

/tmp/ipykernel_693/2747682774.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp"       : datetime.utcnow().isoformat(),



✅ 2889 filas procesadas → '/content/drive/MyDrive/UNSAM/Factor-Data/Proyectos/LLMs_Sillicon_sampling/migraciones/gpt-oss:120B_Q130_v2_WVS_silicon_empirico_results.csv'


## 7. Vista previa y calidad

In [ ]:
# Para recargar sin re-ejecutar la simulación:
# results = pd.read_csv(OUTPUT_CSV)

valid_r = results[results["valid"]].copy()

print(f"Pregunta activa : {results['question'].iloc[0]}")
print(f"Total filas     : {len(results)}")
pct = results['valid'].mean() * 100
print(f"Respuestas válidas: {pct:.1f}%")

results[[
    "country", "sex", "education", "political_ident",
    "empirical", "model", "match", "thinking",
]].head(6)

Pregunta activa : Q130
Total filas     : 2889
Respuestas válidas: 100.0%


,country,sex,education,political_ident,empirical,model,match,thinking
0,Argentina,man,secondary education,center,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,False,I think it's okay for foreigners to come if th...
1,Argentina,man,secondary education,center-right,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,False,I think letting anyone come who wants to is to...
2,Argentina,woman,secondary education,center,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,False,I think letting anyone come who wants to could...
3,Argentina,woman,secondary education,far-right,Place strict limits on the number of foreigner...,Place strict limits on the number of foreigner...,True,I don't want unrestricted immigration because ...
4,Argentina,man,secondary education,far-right,Place strict limits on the number of foreigner...,Place strict limits on the number of foreigner...,True,Let anyone come who wants to – I would reject ...
5,Argentina,woman,secondary education,center,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,False,"I need a job, so I don't want more competition..."


## 8. Distribución por pregunta

In [ ]:
def plot_distribution_by_country(question_code: str, results_df: pd.DataFrame):
    """Plots the empirical and model distribution by country for a given question."""

    if question_code not in QUESTION_PLOTTING_CONFIG:
        print(f"No plotting configuration found for question: {question_code}")
        return

    config = QUESTION_PLOTTING_CONFIG[question_code]
    plot_order = config["order"]
    short_map = config["short_map"]
    title_suffix = config["title_suffix"]

    results_for_plot = results_df.copy()
    if short_map:
        results_for_plot["empirical_display"] = results_for_plot["empirical"].map(short_map)
        results_for_plot["model_display"] = results_for_plot["model"].map(short_map)
    else:
        results_for_plot["empirical_display"] = results_for_plot["empirical"]
        results_for_plot["model_display"] = results_for_plot["model"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
    for ax, country in zip(axes, COUNTRIES):
        sub = results_for_plot[results_for_plot["country"] == country]
        emp = pct_dist(sub["empirical_display"], plot_order)
        model = pct_dist(sub["model_display"], plot_order)
        x = range(len(plot_order))
        w = 0.35
        ax.bar([i - w/2 for i in x], emp, width=w, label="Empírico", color="#546E7A", alpha=0.85)
        ax.bar([i + w/2 for i in x], model, width=w, label="Modelo", color="#E64A19", alpha=0.85)
        ax.set_xticks(list(x))
        if short_map:
            ax.set_xticklabels(plot_order, fontsize=7, rotation=15, ha="right")
        else:
            ax.set_xticklabels([o.replace(" ", "\n") for o in plot_order], fontsize=7)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_title(country, fontsize=12, fontweight="bold")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f"{question_code} — {title_suffix}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f"distribucion_{question_code.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plot_distribution_by_country(QUESTION, results)

NameError: name 'QUESTION_PLOTTING_CONFIG' is not defined

In [ ]:
results

,empirical_id,country,question,sex,education,class,occupation,political_ident,empirical,model,thinking,match,valid,raw_response,backend,model_name,temperature,timestamp
0,1005,Argentina,Q130,man,secondary education,lower middle class,skilled worker,center,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,I think it's okay for foreigners to come if th...,False,True,"{\n ""thinking"": ""I think it's okay for foreig...",ollama,gpt-oss:120B,1.0,2026-05-18T13:16:38.827819
1,1007,Argentina,Q130,man,secondary education,lower middle class,skilled worker,center-right,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,I think letting anyone come who wants to is to...,False,True,"{\n ""thinking"": ""I think letting anyone come ...",ollama,gpt-oss:120B,1.0,2026-05-18T13:16:40.798828
2,1008,Argentina,Q130,woman,secondary education,lower middle class,person who never had a job,center,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,I think letting anyone come who wants to could...,False,True,"{\n ""thinking"": ""I think letting anyone come ...",ollama,gpt-oss:120B,1.0,2026-05-18T13:16:43.105288
3,1010,Argentina,Q130,woman,secondary education,lower middle class,unskilled worker,far-right,Place strict limits on the number of foreigner...,Place strict limits on the number of foreigner...,I don't want unrestricted immigration because ...,True,True,"{\n ""thinking"": ""I don't want unrestricted im...",ollama,gpt-oss:120B,1.0,2026-05-18T13:16:45.522747
4,1011,Argentina,Q130,man,secondary education,lower middle class,semi-skilled worker,far-right,Place strict limits on the number of foreigner...,Place strict limits on the number of foreigner...,Let anyone come who wants to – I would reject ...,True,True,"{\n ""thinking"": ""Let anyone come who wants to...",ollama,gpt-oss:120B,1.0,2026-05-18T13:16:47.318384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2884,92325,United States,Q130,woman,higher education,working class,sales worker,center,Let anyone come who wants to,Let people come as long as there are jobs avai...,Let anyone come who wants to: Too unrestricted...,False,True,"{\n ""thinking"": ""Let anyone come who wants to...",ollama,gpt-oss:120B,1.0,2026-05-18T14:46:06.669405
2885,92327,United States,Q130,man,higher education,lower middle class,professional or technical worker,center-right,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,I think immigration should be allowed only whe...,False,True,"{\n ""thinking"": ""I think immigration should b...",ollama,gpt-oss:120B,1.0,2026-05-18T14:46:08.307051
2886,92334,United States,Q130,woman,higher education,upper middle class,professional or technical worker,center-left,Let people come as long as there are jobs avai...,Let people come as long as there are jobs avai...,I believe immigration can benefit the economy ...,True,True,"{\n ""thinking"": ""I believe immigration can be...",ollama,gpt-oss:120B,1.0,2026-05-18T14:46:10.369372
2887,92337,United States,Q130,woman,higher education,lower middle class,sales worker,center-right,Place strict limits on the number of foreigner...,Let people come as long as there are jobs avai...,"I think immigration should be allowed, but onl...",False,True,"{\n ""thinking"": ""I think immigration should b...",ollama,gpt-oss:120B,1.0,2026-05-18T14:46:12.030993
